# Code for Classification of toric colorable seeds of Picard number $4$
In terms of:
    - Positivity
    - Fan-givingness
    - Projectivity

In [1]:
import SimplicialComplex as sc
import json
import sympy as sp
import numpy as np
from IPython.display import display
from itertools import combinations,permutations

def read_file(filename):
    with open(filename, 'rb') as f:
        data = f.readlines()
        data = [x.strip() for x in data]
    return data

### Function for loading all the seeds of a given pair $(m,n)$

In [51]:


def load_seeds(n,m):
    db_path = 'final_results/CSPLS_%d_%d' % (n, m)
    list_facets = [json.loads(facets_bytes) for facets_bytes in read_file(db_path)]
    return [sc.PureSimplicialComplex(facets_set) for facets_set in list_facets]

def find_orientation(K:sc.PureSimplicialComplex):
    (m,n) = (K.m,K.n)
    orientation = np.zeros(len(K.facets_bin))
    orientation[0]=1
    while np.any(orientation == 0):
        nonoriented = np.where(orientation==0)[0]
        oriented = np.where(orientation!=0)[0]
        for i in nonoriented:
            facet_bin_1 = K.facets_bin[i]
            for j in oriented:
                facet_bin_2 = K.facets_bin[j]
                vertices = sc.binary_to_face_0(facet_bin_1^facet_bin_2,m)
                if len(vertices)==2:
                    vertices = sc.binary_to_face_0(facet_bin_1^facet_bin_2,m)
                    v1=vertices[0]
                    facet1 = sc.binary_to_face_0(facet_bin_1,m)
                    v2=vertices[1]
                    facet2 = sc.binary_to_face_0(facet_bin_2,m)
                    if v1 not in sc.binary_to_face_0(facet_bin_1,m):
                        v1, v2 = v2, v1
                    orientation[i]=(-1)**(facet2.index(v2)+facet1.index(v1)+1)*orientation[j]
                    break
    return(orientation)
    
def compute(K:sc.PureSimplicialComplex,orientation,char_map_skeleton, filter,variable):
    (m,n) = (K.m,K.n)
    # Compute char. map
    location = np.where(filter==False)
    nbr_ind = len(location[0])
    x_vars = sp.symarray(variable,nbr_ind)
    poly_char_map = sp.Matrix(char_map_skeleton)
    k=0
    for k in range(nbr_ind):
        poly_char_map[location[0][k],location[1][k]] = x_vars[k]
    N = len(K.facets_bin)
    is_positive = True
    list_rel=[]
    for k in range(1,N):
        rel = poly_char_map[:,sc.binary_to_face_0(K.facets_bin[k],m)].det().expand()-sp.Integer(orientation[k])
        if rel!=0:
            if len(rel.free_symbols)==0:
                is_positive=False
                break
            else: list_rel.append(rel)
    if is_positive:
        A,b = sp.linear_eq_to_matrix(list_rel,x_vars)
        linsol=sp.linsolve((A,b))
        if len(linsol)>0:
            solution_tuple = next(iter(linsol))  # Extract first (and only) solution tuple
            solution_dict = dict(zip(x_vars, solution_tuple))
            new_poly_char_map = poly_char_map.subs(solution_dict)
            # return(new_poly_char_map)
            a= sp.symbols('a')
            for i in range(n):
                for j in range(m):
                    if len(new_poly_char_map[i,j].free_symbols)>0:
                        sol_coeff_int = sp.diophantine(new_poly_char_map[i,j] + a)
                        if len(sol_coeff_int) >0:
                            solution_tuple_local = next(iter(sol_coeff_int))  # Extract first (and only) solution tuple
                            solution_dict_local = dict(zip(new_poly_char_map[i,j].free_symbols, solution_tuple_local))
                            print(solution_tuple_local)
                            # new_poly_char_map[i,j] = new_poly_char_map[i,j].subs(solution_dict_local)
                        else:
                            print("No solutions")
                            return False
        else:
            print("No solutions")
            return False
    else:
        print("No solutions")
        return False

In [52]:
import SimplicialComplex  as sc
dico_seeds ={}
dico_char_maps={}

def fix_ind(char_map_skeleton,filter,i,j,x):
    char_map_skeleton[i,j] = x
    filter[i,j]=True

d_0 = sp.Symbol('d_0')

K_5 = sc.PureSimplicialComplex([[1,2],[1,5],[2,3],[3,4],[4,5]])
dico_seeds[(2,5,0)] = K_5
dico_char_maps[(2,5,0)] = []
dico_char_maps[(2,5,0)].append(sp.Matrix([[1,0,-1,-1,d_0],[0,1,1,0,-1]]))
dico_char_maps[(2,5,0)].append(sp.Matrix([[1,0,-1,-1,0],[0,1,d_0,d_0-1,-1]]))
dico_char_maps[(2,5,0)].append(sp.Matrix([[1,0,-1,-d_0,1-d_0],[0,1,0,-1,-1]]))
dico_char_maps[(2,5,0)].append(sp.Matrix([[1,0,-1,0,1],[0,1,1-d_0,-1,-1]]))
dico_char_maps[(2,5,0)].append(sp.Matrix([[1,0,-1,d_0-1,1],[0,1,1,-d_0,1]]))

nbr_of_fangiving={}
nbr_of_fangiving[(2,6)]=1
nbr_of_fangiving[(2,5)]=1

K_6 = sc.PureSimplicialComplex([[1,2],[1,6],[2,3],[3,4],[4,5],[5,6]])
dico_seeds[(2,6,0)] = K_6
dico_char_maps[(2,6,0)] = []
first_list = []
for char_map_K_5 in dico_char_maps[(2,5,0)]:
    char_map_K_6 = sp.Matrix(np.zeros((2,6),dtype=int))
    for j in range(6):
        if j==0:
            for i in range(2):
                char_map_K_6[i,j] = char_map_K_5[i,j]
        elif j==1:
            char_map_K_6[1,1] = 1
        else:
            char_map_K_6[0,j] = char_map_K_5[0,j-1] - char_map_K_5[1,j-1]
            char_map_K_6[1,j] = char_map_K_5[1,j-1]
    first_list.append(char_map_K_6.copy())

    for v in range(2,6):
        char_map_K_6 = sp.Matrix(np.zeros((2,6),dtype=int))
        for j in range(6):
            if j<v:
                for i in range(2):
                    char_map_K_6[i,j] = char_map_K_5[i,j]
            elif j>v:
                for i in range(2):
                    char_map_K_6[i,j] = char_map_K_5[i,j-1]
            elif j==v:
                for i in range(2):
                    char_map_K_6[i,j] = char_map_K_5[i,j-1] + char_map_K_5[i,j%5]
        first_list.append(char_map_K_6.copy())
N = len(first_list)
for a in range(N):
    is_similar=False
    diff_char_map = sp.Matrix(np.zeros((2,6),dtype=int))
    for b in range(len(dico_char_maps[(2,6,0)])):
        for i in range(2):
            for j in range(6):
                diff_char_map[i,j]= first_list[a][i,j] -dico_char_maps[(2,6,0)][b][i,j]
        if len(diff_char_map.free_symbols)==0:
            is_similar = True
            break
    if not is_similar:
        dico_char_maps[(2,6,0)].append(first_list[a].copy())
del first_list
# for char_map in dico_char_maps[(2,6,0)]:
#     display(char_map)

# dico_char_maps[(2,6,0)].append(sp.Matrix([[1,0,-1,-1,-1,d_0],[0,1,2,1,0,-1]]))
# dico_char_maps[(2,6,0)].append(sp.Matrix([[1,0,-1,2,-1,d_0],[0,1,1,1,0,-1]]))
# dico_char_maps[(2,6,0)].append(sp.Matrix([[1,0,-1,-1,d_0-1,d_0],[0,1,1,0,-1,-1]]))




In [53]:

def rec(K,variable):
    list_char_maps=[]
    (m,n) = (K.m,K.n)
    best_pic=1
    best_v=0
    for v in range(m):
        test = sc.Link_of(K,sc.list_2_pow[v])
        if test.Pic > best_pic and test.is_a_seed():
            best_link = sc.Link_of(K,sc.list_2_pow[v])
            best_pic = best_link.Pic
            best_v=v
    if best_pic!=K.Pic:
        print("Not the same Pic")
        return False
    # here, we relabel to have best_v in the first facet
    new_facet =0
    for facet_bin in K.facets_bin:
        if sc.list_2_pow[best_v]|facet_bin==facet_bin:
            new_facet = facet_bin
            break
    facet= sc.binary_to_face_0(new_facet,m)
    labels = [i for i in facet if i!=best_v]
    labels.append(best_v)
    for i in range(m):
        if i not in labels:
            labels.append(i)

    new_K = sc.PureSimplicialComplex(sc.relabel_facets(K,labels))
    print(new_K.facets)
    link_n = sc.Link_of(new_K,sc.list_2_pow[n-1])
    k0=-1
    for k in range(nbr_of_fangiving[(link_n.n,link_n.m)]):
        L=dico_seeds[(link_n.n,link_n.m,k)]
        old_labels = sc.find_isom(link_n,L)
        if old_labels:
            k0=k
            break
        
    for char_map in dico_char_maps[(link_n.n,link_n.m,k0)]:
        new_char_map = sp.Matrix(np.zeros((n,m),dtype = int))
        filter = np.zeros((n,m),dtype=bool)
        labels_link =[]
        for i in range(m):
            if i!= n-1:
                for facet_bin in new_K.facets_bin:
                    if sc.list_2_pow[i] | sc.list_2_pow[n-1] | facet_bin == facet_bin:
                        labels_link.append(i)
                        break
        for i in range(link_n.m):
            for j in range(n-1):
                new_char_map[j,labels_link[i]] =  char_map[j,old_labels[i]]
                filter[j,labels_link[i]]= True
        new_char_map[n-1,n-1]=1
        filter[:,:n]= True
        poly_char_map = compute(new_K,find_orientation(new_K),new_char_map,filter,variable)
        if poly_char_map:
            display(poly_char_map)
        # if input('Keep?') == "yes":
        #     list_char_maps.append(poly_char_map)
    return list_char_maps
Data_3_7=load_seeds(3,7)
K_0 = Data_3_7[0]
list_char_maps = rec(K_0,'x')


K_1 = Data_3_7[1]
K_2 = Data_3_7[2]
K_3 = Data_3_7[3]

rec(K_3,'x')


[[1, 2, 3], [1, 2, 4], [1, 3, 7], [1, 4, 6], [1, 6, 7], [2, 3, 5], [2, 4, 5], [3, 4, 5], [3, 4, 6], [3, 6, 7]]
(t_0, -t_0 - 1)
(t_0, -t_0)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, t_0 - 1)
(t_0, t_0)
(t_0, -t_0)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, -t_0 - 1)
(t_0, 1 - t_0)
(1 - 2*t_0, t_0)
(t_0, -t_0)
(t_0, 1 - t_0)
(1, 1)
(-1, 0)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, t_0 + 1)
(t_0, t_0 + 2)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, t_0 - 1)
(t_0, t_0)
(t_0, t_0 + 1)
(t_0, t_0)
(2*t_0 - 1, t_0)
(t_0, t_0 + 1)
(t_0, 2 - t_0)
(t_0, t_0 + 1)
(t_0, t_0 + 1)
(t_0, t_0 + 2)
(t_0, t_0 + 1)
(t_0, -t_0)
No solutions
No solutions
No solutions
[[1, 2, 3], [1, 2, 5], [1, 3, 7], [1, 4, 5], [1, 4, 7], [2, 3, 5], [3, 4, 6], [3, 4, 7], [3, 5, 6], [4, 5, 6]]
(t_0, -t_0 - 1)
(t_0, -t_0)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, t_0)
(t_0, t_0 - 1)
(t_0, 1 - t_0)
(t_0, -t_0)
(t_0, t_0)
(t_0, t_0)
(t_0, 1 - t_0)
(t_0, -t_0 - 1)
(t_0, -t_0)
(2, -2)
(1, -2)


NotImplementedError: No solver has been written for cubic_thue.


2-6 -> 1

3-7 -> 10

4-8 -> 11: [2,3,4,5,6,7,8,9,11,12,14] ->10

5-9 -> 16: [6, 7, 8, 10,  11, 18, 19, 20, 28, 47, 53, 76, 92, 95, 97, 108]

6-10 -> 18: [5,6,11,55,72,126,127,188,271,293,309,326,330,352,392,550,583,616]

7-11 -> 9: [12, 162, 210, 247, 355, 359, 556, 1117, 1155]

8-12 -> 1: [25]

Then -> []
